# Lab 6 — Rate-Limiting & Observability Middleware

Difficulty: Intermediate | ~40 min | Requires Lab 4


### Step 0: Install Dependencies

This cell installs every pinned dependency the lab needs. The remaining libraries support the chat endpoint's LLM client and middleware logic — no authentication libraries are required.


In [1]:
!pip install fastapi==0.112.2 pydantic==2.8.2 httpx==0.28.1 python-dotenv==1.2.3 openai==3.5.0



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 1: Imports, API Key, and App Setup

These imports give us everything the rest of the lab needs: FastAPI for the app and its test client, `dotenv` and `os` to read the API key, `time` to measure latency and drive the rate-limit window, `deque` as an efficient timestamp list for the rate limiter, and the OpenAI client for the chat endpoint.


In [2]:
from fastapi import FastAPI, Depends, Header, Request
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from dotenv import load_dotenv
from openai import AsyncOpenAI
from collections import deque
from typing import Annotated
import os, time, json

load_dotenv()

api_key = os.getenv("OPEN_ROUTER_KEY")
if not api_key:
    api_key = input("Open Router API key: ")

conversation_store = {}
app = FastAPI()


### Step 2: Tenant Identification via Request Header

Before we can allocate a rate limit, we need to know whose quota each request counts against. Rather than build a full authentication system (like we did in Lab 5), we read the tenant ID from a simple `X-Tenant-ID` header. This is a deliberate simplification — a real production system would derive identity from a verified source like a JWT, because a plain header can be trivially spoofed. 

Here it's only used for accounting: which tenant's request budget does this call fall under. If the header is missing, the request is assigned the default `"anonymous"` tenant, so unlabeled callers still get rate-limited instead of slipping through uncounted.


In [3]:
def get_tenant_id(x_tenant_id: str = Header(default="anonymous")):
    return x_tenant_id


### Step 3: Tenant-Scoped Session History

Conversation history lives in the in-memory `conversation_store` dict, keyed by the pair `(tenant_id, session_id)`. We use both halves of the key because two tenants might happen to pick the same `session_id` string — without the tenant half they'd share a single history. The composite key keeps every tenant's conversations fully separated from everyone else's.


In [4]:
async def get_session_history(
    tenant_id: Annotated[str, Depends(get_tenant_id)],
    session_id: str,
):
    key = (tenant_id, session_id)
    if key not in conversation_store:
        conversation_store[key] = []
    return conversation_store[key]


### Step 4: The `/chat` Endpoint with LLM Usage Passback

After getting the LLM's answer, the endpoint stores its token usage on `request.state.usage`. This is necessary because the middleware function runs outside the endpoint and has no direct access to what happened inside it — `request.state` is a plain object attached to the request that both sides can read and write, so this is how the endpoint hands its usage numbers over to the middleware for logging.


In [5]:
client = AsyncOpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

async def get_client():
    return client

@app.post("/chat")
async def chat(
    message: str,
    session_id: str,
    history: Annotated[list, Depends(get_session_history)],
    client_dep: Annotated[AsyncOpenAI, Depends(get_client)],
    request: Request,
):
    history.append({"role": "user", "content": message})

    response = await client_dep.chat.completions.create(
        model="openrouter/free",
        messages=history,
    )

    answer = response.choices[0].message.content
    history.append({"role": "assistant", "content": answer})

    # Pass usage data back to middleware via request.state
    if response.usage:
        request.state.usage = {
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,
        }

    return {"answer": answer, "history": history}


### Step 5: The Rate Limiter and Middleware

The rate limiter keeps a `deque` of request timestamps per tenant. The middleware must reject a request *before* the endpoint runs — once the endpoint runs, a billed LLM call has already happened. So the middleware checks the tenant's window first, and if it's already at the threshold it returns 429 and the whole pipeline short-circuits. Otherwise it records the timestamp, lets the endpoint run, and afterward prints a single structured log line with the latency and the token usage it reads from `request.state`. The 429 path and the normal path share one `_log` helper, so every request emits exactly one log line — a rejected line simply carries no usage numbers, because nothing ever ran.


In [6]:
RATE_LIMIT_WINDOW = 45   # seconds
RATE_LIMIT_THRESHOLD = 3  # max requests per tenant in window

rate_limit_store = {}  # tenant_id -> deque of timestamps

def _log(response, tenant_id, request, start_time):
    latency_ms = round((time.time() - start_time) * 1000, 2)
    usage = getattr(request.state, "usage", None)
    log_line = {
        "tenant_id": tenant_id,
        "path": request.url.path,
        "status_code": response.status_code,
        "latency_ms": latency_ms,
    }
    if usage:
        log_line["prompt_tokens"] = usage["prompt_tokens"]
        log_line["completion_tokens"] = usage["completion_tokens"]
        log_line["total_tokens"] = usage["total_tokens"]

    print(json.dumps(log_line))

@app.middleware("http")
async def observability_middleware(request: Request, call_next):
    start_time = time.time()
    tenant_id = None

    # Only rate-limit /chat — the only route that triggers LLM costs
    if request.url.path == "/chat":
        tenant_id = request.headers.get("x-tenant-id", "anonymous")

        now = time.time()
        if tenant_id not in rate_limit_store:
            rate_limit_store[tenant_id] = deque()

        # Drop timestamps outside the window
        window = rate_limit_store[tenant_id]
        while window and window[0] <= now - RATE_LIMIT_WINDOW:
            window.popleft()

        if len(window) >= RATE_LIMIT_THRESHOLD:
            rejected = JSONResponse(
                status_code=429,
                content={
                    "detail": f"Rate limit exceeded for {tenant_id}. "
                    f"Max {RATE_LIMIT_THRESHOLD} requests per "
                    f"{RATE_LIMIT_WINDOW}s window."
                },
            )
            _log(rejected, tenant_id, request, start_time)
            return rejected
        window.append(now)

    # Let the endpoint handle the request
    response = await call_next(request)

    # Post-request: latency, usage, structured log
    _log(response, tenant_id, request, start_time)
    return response


### Step 6: TestClient

A single test client drives every demo. Note: because `/chat` awaits an external async client, running `TestClient` repeatedly in a single notebook session can occasionally raise an "Event loop is closed" error. If that happens, simply re-run the cell — a fresh event loop is created on retry.


In [7]:
test_client = TestClient(app)


### Demo 1: Normal /chat call — under the limit

Send a single message with `X-Tenant-ID: tenant-a`. This is the happy path: one request, well under the limit, so the rate limiter lets it through and the endpoint makes a real LLM call. After the response, the middleware prints the structured log line you'll see in every demo — tenant, path, status code, latency, and the real token counts that the endpoint handed over via `request.state`.


In [8]:
res = test_client.post(
    "/chat",
    params={"message": "What is the capital of France?", "session_id": "demo-session"},
    headers={"X-Tenant-ID": "tenant-a"},
)

print("status:", res.status_code)
print("answer:", res.json()["answer"][:80] + "...")


{"tenant_id": "tenant-a", "path": "/chat", "status_code": 200, "latency_ms": 5917.9, "prompt_tokens": 27, "completion_tokens": 187, "total_tokens": 214}
status: 200
answer: The capital of France is Paris....


### Demo 2: Rapid /chat calls — exceeding the limit

Send several more messages back to back for the same tenant. Each call takes only a few seconds, so they all land inside the same 45-second window. Demo 1 already used up one of tenant-a's three requests, so once the window holds three, the next calls are rejected. Watch the two things that matter here: the 429s come back without the endpoint ever running, and their log lines carry no usage fields — that's the evidence the LLM was never called, which is exactly why the check lives in middleware.


In [9]:
for i, msg in enumerate(["Second call", "Third call", "Fourth call", "Fifth call"]):
    res = test_client.post(
        "/chat",
        params={"message": msg, "session_id": "demo-session"},
        headers={"X-Tenant-ID": "tenant-a"},
    )
    print(f"call {i+1}: status={res.status_code}", end="")
    if res.status_code == 429:
        print(f"  -> {res.json()['detail']}")
    else:
        print()


{"tenant_id": "tenant-a", "path": "/chat", "status_code": 200, "latency_ms": 2150.57, "prompt_tokens": 190, "completion_tokens": 36, "total_tokens": 226}
call 1: status=200
{"tenant_id": "tenant-a", "path": "/chat", "status_code": 200, "latency_ms": 3434.21, "prompt_tokens": 1206, "completion_tokens": 186, "total_tokens": 1392}
call 2: status=200
{"tenant_id": "tenant-a", "path": "/chat", "status_code": 429, "latency_ms": 0.0}
call 3: status=429  -> Rate limit exceeded for tenant-a. Max 3 requests per 45s window.
{"tenant_id": "tenant-a", "path": "/chat", "status_code": 429, "latency_ms": 0.0}
call 4: status=429  -> Rate limit exceeded for tenant-a. Max 3 requests per 45s window.


### Demo 3: Second tenant — per-tenant isolation

Call `/chat` with `X-Tenant-ID: tenant-b` right now, without sleeping. tenant-a's burst from Demo 2 is still inside its window, so its quota is exhausted — yet tenant-b has its own separate rate-limit store, so it succeeds. This is the proof that limits are scoped per tenant: one tenant spending its budget never blocks another.


In [10]:
# tenant-b has a completely separate rate-limit store from tenant-a.
res = test_client.post(
    "/chat",
    params={"message": "Hello from tenant-b — I should not be rate-limited", "session_id": "bob-session"},
    headers={"X-Tenant-ID": "tenant-b"},
)

print("tenant-b status:", res.status_code)
print("tenant-b answer:", res.json()["answer"][:60] + "...")


{"tenant_id": "tenant-b", "path": "/chat", "status_code": 200, "latency_ms": 6993.24, "prompt_tokens": 54, "completion_tokens": 818, "total_tokens": 872}
tenant-b status: 200
tenant-b answer: It sounds like you're working in a multi-tenant environment ...


### Demo 4: Window recovery — the limit resets

Sleep past the 45-second window, then call `/chat` again with tenant-a. The rate limiter drops the timestamps that have fallen out of the window, sees fewer than the threshold, and lets the request through. This proves the limit is a rolling window that recovers over time — not a permanent lockout.


In [11]:
print(f"Sleeping {RATE_LIMIT_WINDOW}s for the window to reset...")
time.sleep(RATE_LIMIT_WINDOW)

res = test_client.post(
    "/chat",
    params={"message": "Window reset — I should work again", "session_id": "demo-session"},
    headers={"X-Tenant-ID": "tenant-a"},
)

print("status:", res.status_code)
print("answer:", res.json()["answer"][:80] + "...")


Sleeping 45s for the window to reset...
{"tenant_id": "tenant-a", "path": "/chat", "status_code": 200, "latency_ms": 9805.01, "prompt_tokens": 126, "completion_tokens": 387, "total_tokens": 513}
status: 200
answer: Window reset acknowledged! I'm ready to help whenever you are. What would you li...
